In [0]:
VOLUME_PATH = "/Volumes/airbnb/default/raw"
spark.sql(f"LIST'{VOLUME_PATH}'").display()

In [0]:
spark.sql(f"LIST'{VOLUME_PATH}/data'").display()

In [0]:
%sql
SHOW SCHEMAS IN supply_chain_demo

In [0]:
%sql 

FROM airbnb.bronze.raw_supply_chain LIMIT 5; 

a) Show a histogram of numbers of reviews for each accomodation.

b) Clean the price column. Is it numeric?

c) Calculate the average price of accomodations per neighborhood group. In which neighborhood group is the average price highest?

d) Show the top 10 accomodations with most reviews.

e) Show the top 10 accomodations with best reviews.

f) Other EDAs of your choice


In [0]:
# A
# read the scheam
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')           # Hanterar komman inuti citattecken
    .option("escape", '"')          # Hanterar citerade citattecken
    .option("multiLine", True)      # Viktigt för beskrivningar med radbrytningar
    .load(f"{VOLUME_PATH}/data/Airbnb_Open_Data.csv")
)

df.createOrReplaceTempView("hosts_data")

# print the columns

print(df.columns)

In [0]:
num_of_reviews = spark.sql("""
          SELECT
            coalesce(`NAME`, 'No Name') NAME,
            `number of reviews`
          
          FROM 
            hosts_data
          """)

num_of_reviews.display()



In [0]:
import plotly.express as px

pdf = num_of_reviews.toPandas()

fig = px.histogram(
    pdf, 
    x="number of reviews", 
    title="Histogram of numbers of reviews for each accomodationr",
    labels="Number of reviews",
    color_discrete_sequence=["#228B22"] # Skogsgrön färg, välj vad du vill!
)

# 3. Uppdatera layouten för att göra det ännu snyggare
fig.update_layout(
    bargap=0.1, 
    xaxis_title="Num of reviews",
    yaxis_title="Num of accomendations"
)

fig.show()

In [0]:
from pyspark.sql.functions import col, translate, regexp_replace

cleaned_df = df.withColumn(
    "price", 
    regexp_replace(col("price"), "[$, ]", "").cast("float")
).withColumn(
    "service fee", 
    regexp_replace(col("service fee"), "[$, ]", "").cast("float")
).withColumn(
    "number of reviews", 
    col("number of reviews").cast("int")
)

cleaned_df.createOrReplaceTempView("cleaned")

In [0]:
avg_price_by_neighbourhood = spark.sql("""
          SELECT
            neighbourhood,
            ROUND(AVG(`price`),2) as avg_price
            
          FROM 
            cleaned  
          GROUP BY neighbourhood
          ORDER BY avg_price DESC
          """)
avg_price_by_neighbourhood.display()


In [0]:
top_10_accomodattions_reviews = spark.sql("""
          SELECT
            `number of reviews`,
            NAME
          FROM 
            cleaned  
          GROUP BY `number of reviews`, NAME
          ORDER BY `number of reviews` DESC
          LIMIT 10
          """)
top_10_accomodattions_reviews.display()


In [0]:
top_10_accomodattions_best_reviews = spark.sql("""
          SELECT
            `review rate number`,
            `number of reviews`,
            NAME    
          FROM 
            cleaned  
          GROUP BY `review rate number`, `number of reviews`, NAME
          ORDER BY `review rate number` DESC
          LIMIT 10
          """)
top_10_accomodattions_best_reviews.display()